# Flight Ticket Price Prediction - Model Building & Model Selection

## Objective

The objective of this notebook is to train, evaluate, and compare multiple machine learning regression models to predict flight ticket prices.

In this notebook, we will:

- Load the processed dataset
- Define evaluation metrics
- Train multiple regression models
- Compare model performance
- Select the best-performing model
- Save the final trained model

The workflow follows industry-standard machine learning practices and ensures reproducible model development.

In [6]:
# =====================================================
# Import Required Libraries
# =====================================================

# Data Manipulation
import pandas as pd
import numpy as np

# Machine Learning Models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

# Model Evaluation
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Hyperpaarameter Tuning 
from sklearn.model_selection import GridSearchCV

# Save Model
import joblib

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

print("Libraries Imported Successfully")

Libraries Imported Successfully


# Load Dataset and Preprocessing Pipeline

## Objective

In this step, we load the cleaned dataset and the saved preprocessing pipeline.

The workflow includes:

- Loading the cleaned dataset
- Loading the saved preprocessing pipeline
- Creating feature matrix (X) and target variable (y)
- Performing Train-Test Split
- Applying the preprocessing pipeline

This ensures that the exact same preprocessing used during training is consistently applied before model building.

In [9]:
# =====================================================
# Load Dataset & Preprocessing Pipeline
# =====================================================

# Load Dataset
data = pd.read_csv("Dataset/Clean_Dataset.csv")

# Remove unnecessary column
if "flight" in data.columns:
    data.drop(columns="flight", inplace=True)

# Features & Target
X = data.drop(columns="price")
y = data["price"]

# Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Load saved Prprocessor
preprocessor = joblib.load("artifacts/preprocessor.pkl")

# Apply Preprocessing
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("=" * 60)
print("Dataset Loaded Successfully.\n")

print(f"Training Shape : {X_train_processed.shape}")
print(f"Testing Shape : {X_test_processed.shape}")

Dataset Loaded Successfully.

Training Shape : (240122, 37)
Testing Shape : (60031, 37)


# Define Evaluation Metrics

## Objective

Before training machine learning models, we define the evaluation metrics that will be used to compare their performance.

For this regression problem, we will use:

- R² Score
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)

These metrics help us evaluate how accurately a model predicts flight ticket prices.

In [16]:
# =====================================================
# Define Evaluation Function
# =====================================================

def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Train Model
    model.fit(X_train, y_train)

    # Predictions
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)

    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    # Return Results
    return {
        "Model": model.__class__.__name__,
        "Train R²": round(train_r2, 4),
        "Test R²": round(test_r2, 4),
        "Train MAE": round(train_mae, 2),
        "Test MAE": round(test_mae, 2),
        "Train RMSE": round(train_rmse, 2),
        "Test RMSE": round(test_rmse, 2)
    }

print("Model Evaluation Function Created Successfully...")

Model Evaluation Function Created Successfully...


# Baseline Model (Dummy Regressor)

## Objective

Before training complex machine learning models, it is important to establish a baseline performance.

A **Dummy Regressor** does not learn any relationship between the features and the target variable. Instead, it predicts a constant value (by default, the mean of the target variable).

The baseline model helps us answer the following question:

> **Are our machine learning models actually learning useful patterns, or are they only performing as well as a simple constant prediction?**

Any model that cannot outperform the baseline is not considered useful for deployment.

In [17]:
# =====================================================
# Baseline Model (Dummy Regressor)
# =====================================================

# Create Baseline Model
baseline_model = DummyRegressor(strategy="mean")

# Evaluate Baseline Model
baseline_results = evaluate_model(
    baseline_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

# Display Results
baseline_df = pd.DataFrame([baseline_results])

display(baseline_df)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,DummyRegressor,0.0,-0.0,19757.18,19768.69,22696.1,22704.24


# Linear Regression

## Objective

Linear Regression is the simplest and most interpretable regression algorithm.

It models the relationship between input features and the target variable by fitting a linear equation.

In this step, we will:

- Train a Linear Regression model
- Evaluate its performance
- Compare it with the baseline model

This model serves as a benchmark before moving to more complex algorithms.

In [18]:
# =====================================================
# Linear Regression
# =====================================================

# Create Model
linear_model = LinearRegression()

# Evaluate Model
linear_results = evaluate_model(
    linear_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

# Display Results
linear_df = pd.DataFrame([linear_results])

display(linear_df)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,LinearRegression,0.9115,0.9113,4573.98,4553.29,6752.14,6761.71


# Ridge Regression

## Objective

Ridge Regression is an extension of Linear Regression that applies **L2 Regularization**.

Regularization helps reduce model complexity by shrinking the coefficient values without completely removing any feature.

Benefits of Ridge Regression:

- Reduces overfitting
- Handles multicollinearity
- Improves model generalization
- Produces more stable predictions

In this step, we train and evaluate a Ridge Regression model and compare its performance with Linear Regression.

In [19]:
# =====================================================
# Ridge Regression
# =====================================================

# Create Model
ridge_model = Ridge(alpha=1.0)

# Evaluate Model
ridge_results = evaluate_model(
    ridge_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

# Display Results
ridge_df = pd.DataFrame([ridge_results])

display(ridge_df)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Ridge,0.9115,0.9113,4573.91,4553.22,6752.14,6761.72


# Lasso Regression

## Objective

Lasso Regression is an extension of Linear Regression that uses **L1 Regularization**.

Unlike Ridge Regression, Lasso can reduce some feature coefficients to exactly zero, effectively performing automatic feature selection.

Benefits of Lasso Regression:

- Reduces overfitting
- Performs feature selection
- Removes less important features
- Produces simpler and more interpretable models

In this step, we train and evaluate a Lasso Regression model and compare its performance with other linear models.

In [20]:
# =====================================================
# Lasso Regression
# =====================================================

# Create Model
lasso_model = Lasso(alpha=1.0)

# Evaluate Model
lasso_results = evaluate_model(
    lasso_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

# Display Results
lasso_df = pd.DataFrame([lasso_results])

display(lasso_df)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,Lasso,0.9115,0.9113,4572.17,4551.58,6752.18,6761.8


# Decision Tree Regressor

## Objective

Decision Tree Regression is a non-linear machine learning algorithm that predicts continuous values by recursively splitting the dataset into smaller subsets.

Unlike Linear Regression, Decision Trees can learn complex and non-linear relationships between features and the target variable.

Advantages:

- Captures non-linear relationships
- Easy to understand and interpret
- Handles both numerical and categorical features (after preprocessing)
- Does not assume linearity

However, Decision Trees can easily overfit the training data if not properly controlled.

In this step, we train and evaluate a Decision Tree Regressor and compare its performance with previous models.

In [21]:
# =====================================================
# Decision Tree Regressor
# =====================================================

# Create Model
decision_tree_model = DecisionTreeRegressor(
    random_state=42,
    max_depth=10
)

# Evaluate Model
decision_tree_results = evaluate_model(
    decision_tree_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

# Display Results
decision_tree_df = pd.DataFrame([decision_tree_results])

display(decision_tree_df)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,DecisionTreeRegressor,0.962,0.9603,2435.46,2478.8,4421.59,4525.89


# Random Forest Regressor

## Objective

Random Forest is an ensemble learning algorithm that combines multiple Decision Trees to improve prediction accuracy and reduce overfitting.

Instead of relying on a single tree, Random Forest builds many independent trees and combines their predictions.

Advantages:

- High prediction accuracy
- Reduces overfitting
- Handles non-linear relationships
- Robust to noise and outliers
- One of the most widely used algorithms in industry

In this step, we train and evaluate a Random Forest Regressor and compare its performance with previous models.

In [22]:
# =====================================================
# Random Forest Regressor
# =====================================================

# Create Model
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

# Evaluate Model
random_forest_results = evaluate_model(
    random_forest_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

# Display Results
random_forest_df = pd.DataFrame([random_forest_results])

display(random_forest_df)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,RandomForestRegressor,0.9814,0.9769,1611.56,1782.35,3094.35,3450.52


# Extra Trees Regressor

## Objective

Extra Trees Regressor (Extremely Randomized Trees) is an ensemble learning algorithm similar to Random Forest.

Unlike Random Forest, Extra Trees introduces more randomness while building trees by selecting split points randomly.

This additional randomness often results in:

- Faster training
- Better generalization
- Reduced variance
- Lower risk of overfitting

In this step, we train and evaluate an Extra Trees Regressor and compare its performance with other regression models.

In [23]:
# =====================================================
# Extra Trees Regressor
# =====================================================

# Create Model
extra_trees_model = ExtraTreesRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

# Evaluate Model
extra_trees_results = evaluate_model(
    extra_trees_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

# Display Results
extra_trees_df = pd.DataFrame([extra_trees_results])

display(extra_trees_df)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,ExtraTreesRegressor,0.9782,0.9746,1728.51,1868.11,3350.42,3615.9


# Gradient Boosting Regressor

## Objective

Gradient Boosting Regressor is an ensemble learning algorithm that builds multiple Decision Trees sequentially.

Unlike Random Forest, where trees are built independently, Gradient Boosting trains each new tree to correct the errors made by the previous trees.

Advantages:

- Captures complex non-linear relationships
- High predictive accuracy
- Reduces prediction errors iteratively
- Performs well on structured/tabular datasets

In this step, we train and evaluate a Gradient Boosting Regressor and compare its performance with previous models.

In [24]:
# =====================================================
# Gradient Boosting Regressor
# =====================================================

# Create Model
gradient_boosting_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Evaluate Model
gradient_boosting_results = evaluate_model(
    gradient_boosting_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

# Display Results
gradient_boosting_df = pd.DataFrame([gradient_boosting_results])

display(gradient_boosting_df)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,GradientBoostingRegressor,0.9525,0.9515,2944.32,2947.88,4948.76,4998.02


# XGBoost Regressor

## Objective

XGBoost (Extreme Gradient Boosting) is an optimized implementation of the Gradient Boosting algorithm.

It is designed for high performance, scalability, and accuracy.

Advantages:

- Faster training than traditional Gradient Boosting
- Better prediction accuracy
- Built-in regularization
- Handles missing values efficiently
- Widely used in Kaggle competitions and production ML systems

In this step, we train and evaluate an XGBoost Regressor and compare its performance with other machine learning models.

In [ ]:
# =====================================================
# Import XGBoost
# =====================================================

from xgboost import XGBRegressor

print("XGBoost Imported Successfully.")

XGBoost Imported Successfully.


In [26]:
# =====================================================
# XGBoost Regressor
# =====================================================

# Create Model
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Evaluate Model
xgb_results = evaluate_model(
    xgb_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

# Display Results
xgb_df = pd.DataFrame([xgb_results])

display(xgb_df)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,XGBRegressor,0.9688,0.9674,2311.84,2341.87,4012.02,4099.26


# Model Performance Comparison

## Objective

After training multiple regression models, it is essential to compare their performance using the same evaluation metrics.

This comparison helps identify:

- The best-performing model
- Models suffering from overfitting
- Models with good generalization
- The candidate model for Hyperparameter Tuning

The final model will be selected based on its performance on the test dataset rather than the training dataset.

In [27]:
# =====================================================
# Model Comparison
# =====================================================

# Combine Results
model_results = pd.DataFrame([
    baseline_results,
    linear_results,
    ridge_results,
    lasso_results,
    decision_tree_results,
    random_forest_results,
    extra_trees_results,
    gradient_boosting_results,
    xgb_results
])

# Sort by Test R² Score
model_results = model_results.sort_values(
    by="Test R²",
    ascending=False
).reset_index(drop=True)

# Display Comparison Table
display(model_results)

,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,RandomForestRegressor,0.9814,0.9769,1611.56,1782.35,3094.35,3450.52
1,ExtraTreesRegressor,0.9782,0.9746,1728.51,1868.11,3350.42,3615.90
2,XGBRegressor,0.9688,0.9674,2311.84,2341.87,4012.02,4099.26
3,DecisionTreeRegressor,0.9620,0.9603,2435.46,2478.80,4421.59,4525.89
4,GradientBoostingRegressor,0.9525,0.9515,2944.32,2947.88,4948.76,4998.02
5,LinearRegression,0.9115,0.9113,4573.98,4553.29,6752.14,6761.71
6,Ridge,0.9115,0.9113,4573.91,4553.22,6752.14,6761.72
7,Lasso,0.9115,0.9113,4572.17,4551.58,6752.18,6761.80
8,DummyRegressor,0.0000,-0.0000,19757.18,19768.69,22696.10,22704.24


# Hyperparameter Tuning (Best Model)

## Objective

The default parameters of a machine learning model may not provide the best performance.

Hyperparameter Tuning is the process of searching for the optimal combination of parameters that maximizes model performance.

In this notebook, we use **GridSearchCV** to automatically evaluate multiple parameter combinations using Cross Validation.

Benefits:

- Better model accuracy
- Improved generalization
- Reduced overfitting
- Automated parameter selection

In [28]:
# =====================================================
# Hyperparameter Tuning (XGBoost)
# =====================================================

# Parameter Grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# Base Model
xgb = XGBRegressor(
    random_state=42
)

# Grid Search
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Train
grid_search.fit(
    X_train_processed,
    y_train
)

print("Hyperparameter Tuning Completed.")

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Hyperparameter Tuning Completed.


# Best Model Evaluation

## Objective

After Hyperparameter Tuning, we evaluate the best model selected by GridSearchCV.

This step includes:

- Displaying the best hyperparameters
- Displaying the best cross-validation score
- Evaluating the tuned model on the test dataset
- Comparing the tuned model with the original model

The objective is to verify whether Hyperparameter Tuning has improved the model's performance before deployment.

In [29]:
# =====================================================
# Best Model Evaluation
# =====================================================

# Best Model
best_model = grid_search.best_estimator_

# Best Parameters
print("=" * 60)
print("Best Hyperparameters")
print(grid_search.best_params_)

print("\nBest Cross Validation R² Score")
print(round(grid_search.best_score_, 4))

print("=" * 60)

# Evaluate Best Model
best_model_results = evaluate_model(
    best_model,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

best_model_df = pd.DataFrame([best_model_results])

display(best_model_df)

Best Hyperparameters
{'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 8, 'n_estimators': 200, 'subsample': 0.8}

Best Cross Validation R² Score
0.983


,Model,Train R²,Test R²,Train MAE,Test MAE,Train RMSE,Test RMSE
0,XGBRegressor,0.9857,0.9829,1514.8,1626.43,2712.59,2969.65


# Save Final Model & Project Artifacts

## Objective

The final step of the model building pipeline is to save all important artifacts required for deployment.

The following artifacts will be saved:

- Best Trained Model
- Model Comparison Report
- Best Hyperparameters

These files will be used during deployment so that retraining is not required.

In [30]:
# =====================================================
# Save Final Model & Project Artifacts
# =====================================================

import json
import os

# Create artifacts directory
os.makedirs("artifacts", exist_ok=True)

# Save Best Model
joblib.dump(best_model, "artifacts/best_model.pkl")

# Save Model Comparison
model_results.to_csv(
    "artifacts/model_comparison.csv",
    index=False
)

# Save Best Parameters
with open("artifacts/best_parameters.json", "w") as file:
    json.dump(
        grid_search.best_params_,
        file,
        indent=4
    )

print("=" * 60)
print("All Artifacts Saved Successfully")
print("=" * 60)

print("\nSaved Files:")
print("✔ best_model.pkl")
print("✔ model_comparison.csv")
print("✔ best_parameters.json")

All Artifacts Saved Successfully

Saved Files:
✔ best_model.pkl
✔ model_comparison.csv
✔ best_parameters.json
